# MoLE: Mixture-of-LoRA-Experts Router on Google Colab

Trains a per-layer router on top of three already-trained single-task LoRA adapters (E2E, DART, WebNLG). Base GPT-2 Medium and the three LoRA experts stay frozen — only the router parameters are trainable. After training, the routed model is evaluated on each task's test set, plus an ablation that pins the gate to one expert at a time.

Recommended runtime: `Runtime > Change runtime type > GPU` (T4 is sufficient; the router is tiny).

This notebook assumes you have already run `colab_train_lora.ipynb` (E2E), `colab_train_lora_dart.ipynb`, and `colab_train_lora_webnlg.ipynb`, and the resulting `adapter_final.pt` files have been backed up to Drive at:

```
/content/drive/MyDrive/{e2e,dart,webnlg}_lora_r4_alpha32/checkpoints/adapter_final.pt
```

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone Or Update The Repo

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "mole-experiment"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())
!git log --oneline -3

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive

Three reads (one per single-task adapter) and one write (the trained router) all flow through Drive so this notebook can be re-run without retraining the experts.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/mole_e2e_dart_webnlg')
LOCAL_RUN_DIR = Path('outputs/runs/mole_e2e_dart_webnlg')
LOCAL_ROUTER = LOCAL_RUN_DIR / 'checkpoints' / 'router_final.pt'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)


def backup_to_run(relative_path, destination_name=None):
    source = Path(relative_path)
    if not source.exists():
        print('skip missing:', source)
        return None
    destination = DRIVE_RUN_DIR / (destination_name or source.name)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print('backed up:', source, '->', destination)
    return destination

print('Drive run dir:', DRIVE_RUN_DIR)
print('Local run dir:', LOCAL_RUN_DIR)

## 5. Pull Single-Task Expert Adapters From Drive

The MoLE config (`configs/mole_e2e_dart_webnlg.yaml`) expects each adapter at the path inside the project working tree, e.g. `e2e_lora_r4_alpha32/checkpoints/adapter_final.pt`. We copy each from Drive into that local layout so the config resolves correctly without edits.

In [ ]:
import shutil
from pathlib import Path

EXPERTS = [
    ('e2e_lora_r4_alpha32',    Path('/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_final.pt')),
    ('dart_lora_r4_alpha32',   Path('/content/drive/MyDrive/dart_lora_r4_alpha32/checkpoints/adapter_final.pt')),
    ('webnlg_lora_r4_alpha32', Path('/content/drive/MyDrive/webnlg_lora_r4_alpha32/checkpoints/adapter_final.pt')),
]

for run_name, src in EXPERTS:
    dst_dir = Path(run_name) / 'checkpoints'
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / 'adapter_final.pt'
    if not src.exists():
        raise FileNotFoundError(f'expert checkpoint missing in Drive: {src}. Run the per-task notebook first.')
    shutil.copy2(src, dst)
    print(f'{run_name}: {dst}  ({dst.stat().st_size / 1e6:.1f} MB)')

## 6. Download + Tokenize Per-Task Data

The MoLE router trains on the union of the three task training sets. Earlier versions of this notebook tried to pull pre-tokenized JSONL from Drive, but the per-task Colab notebooks only back up `outputs/runs/...` (not `data/processed/`), so that path doesn't exist on most users' Drives. This cell instead downloads the raw E2E / DART / WebNLG splits and runs each task's prep + tokenize step inline. Total network is ~30 MB; tokenization takes a couple of minutes for ~80k examples per task.

In [ ]:
from pathlib import Path

# 6a. E2E raw splits (from the official Microsoft LoRA NLG examples).
!mkdir -p data/raw/e2e
!curl -L -s -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -s -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -s -o data/raw/e2e/test.txt  https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt

# 6b. DART v1.1.1 (Yale-LILY release).
!mkdir -p data/raw/dart
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-train.json https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-train.json
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-dev.json   https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-dev.json
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-test.json  https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-test.json

# 6c. WebNLG (the version the official LoRA examples use).
!mkdir -p data/raw/webnlg
!curl -L -s -o data/raw/webnlg/train.json https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/train.json
!curl -L -s -o data/raw/webnlg/dev.json   https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/dev.json
!curl -L -s -o data/raw/webnlg/test.json  https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/test.json

# 6d. Triple-linearization for DART and WebNLG (E2E is already context||completion text).
!python scripts/prepare_dart_raw.py    --config configs/dart_gpt2_medium_lora.yaml
!python scripts/prepare_webnlg_raw.py  --config configs/webnlg_gpt2_medium_lora.yaml

# 6e. Tokenize all three with the same gpt2-medium tokenizer the experts were trained with.
!python scripts/prepare_e2e.py  --config configs/e2e_gpt2_medium_lora.yaml
!python scripts/prepare_e2e.py  --config configs/dart_gpt2_medium_lora.yaml
!python scripts/prepare_e2e.py  --config configs/webnlg_gpt2_medium_lora.yaml

# Sanity: every task should have train/valid JSONL and a references_test file.
!ls -lh data/processed/e2e_gpt2/{train,valid,test}.jsonl
!ls -lh data/processed/dart_gpt2/{train,valid,test}.jsonl data/processed/dart_gpt2/references_test.txt
!ls -lh data/processed/webnlg_gpt2/{train,valid,test}.jsonl data/processed/webnlg_gpt2/references_test.txt

## 7. Run Tests

In [ ]:
!python -m pytest -q tests/test_mole.py

## 8. Dry Run

Loads GPT-2 Medium, injects MoLE, copies in the three expert adapters, and prints the router parameter count without taking any optimizer steps. This is the cheapest way to verify the expert checkpoints align with the configured rank/alpha/targets.

In [ ]:
!python scripts/train_mole.py --config configs/mole_e2e_dart_webnlg.yaml

## 9. Train The Router

Optimizes only the per-layer router parameters on the union of the three task train splits. With LR `5e-4`, batch `8`, 2 epochs (router-only is much cheaper than full LoRA training).

In [ ]:
!python scripts/train_mole.py --config configs/mole_e2e_dart_webnlg.yaml --train --device cuda

## 10. Inspect Router Training Logs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

metrics_path = Path('outputs/runs/mole_e2e_dart_webnlg/metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
print(f'{len(records)} metric records total')

# 1. Mixed val loss curve
val = [r for r in records if r.get('type') == 'validation']
print('\n--- mixed train/val loss per epoch ---')
for r in val:
    print(f"epoch {r['epoch']:>2}: train {r['train_loss']:.4f}  val {r['val_loss']:.4f}")

# 2. Per-task val loss (collapse-symptom: if one task's loss diverges
#    while the mixed mean still trends down, the gate is starving it)
per_task = defaultdict(list)
for r in records:
    if r.get('type') == 'validation_per_task':
        per_task[r['task']].append((r['epoch'], r['val_loss']))
print('\n--- per-task val loss per epoch ---')
for task, series in per_task.items():
    losses = ', '.join(f'e{e}: {l:.4f}' for e, l in series)
    print(f'{task:>8}: {losses}')

# 3. Per-task routing distribution (collapse detector: each row should
#    show distinct argmax_fraction patterns; if every row is the same,
#    the gate is task-blind)
print('\n--- per-task routing distribution (final epoch) ---')
final_epoch = max(r['epoch'] for r in records if r.get('type') == 'validation')
expert_names = ['e2e', 'dart', 'webnlg']
print(f'{"task":>8} | {"mean softmax":>30} | {"argmax fraction":>30} | entropy')
for r in records:
    if r.get('type') == 'routing_per_task' and r['epoch'] == final_epoch:
        agg = r['aggregate']
        ms = ' '.join(f'{p:.2f}' for p in agg['mean_softmax'])
        af = ' '.join(f'{p:.2f}' for p in agg['argmax_fraction'])
        print(f"{r['task']:>8} | {ms:>30} | {af:>30} | {agg['mean_entropy']:.3f}")

!ls -lh outputs/runs/mole_e2e_dart_webnlg/checkpoints

## 10b. Per-Layer Routing Heatmap (Optional)

Plot per-task argmax fraction across all 24 transformer blocks. A healthy router shows distinct task→expert mappings (e.g. on E2E rows, expert 0 dominates most blocks; on DART rows, expert 1 dominates). Uniform fractions across all blocks indicate the gate didn't learn to discriminate.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

metrics_path = Path('outputs/runs/mole_e2e_dart_webnlg/metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
final_epoch = max(r['epoch'] for r in records if r.get('type') == 'validation')

routing_by_task = {
    r['task']: r['per_layer']
    for r in records
    if r.get('type') == 'routing_per_task' and r['epoch'] == final_epoch
}

if not routing_by_task:
    print('no per-task routing records; skipping heatmap.')
else:
    expert_names = ['e2e', 'dart', 'webnlg']
    tasks = list(routing_by_task.keys())
    fig, axes = plt.subplots(len(tasks), 1, figsize=(10, 2.0 * len(tasks)), sharex=True)
    if len(tasks) == 1:
        axes = [axes]
    for ax, task in zip(axes, tasks):
        layer_keys = sorted(int(k) for k in routing_by_task[task].keys())
        matrix = np.array([
            routing_by_task[task][str(L) if str(L) in routing_by_task[task] else L]['argmax_fraction']
            for L in layer_keys
        ]).T
        im = ax.imshow(matrix, aspect='auto', vmin=0, vmax=1, cmap='viridis')
        ax.set_yticks(range(len(expert_names)))
        ax.set_yticklabels(expert_names)
        ax.set_ylabel(f'task: {task}')
    axes[-1].set_xlabel('transformer layer')
    fig.colorbar(im, ax=axes, label='argmax fraction')
    plt.suptitle(f'Per-task routing argmax fractions (epoch {final_epoch})')
    plt.show()

## 11. Back Up Trained Router To Drive

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/mole_e2e_dart_webnlg.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_mole.ipynb')

print('Trained router backed up to:', DRIVE_RUN_DIR)
!find /content/drive/MyDrive/mole_e2e_dart_webnlg -maxdepth 2 -type f | sort | tail -20

## 12. Generate Per-Task Test Predictions Through The Router

Each task uses its own data/decoding config; the routed model is constructed once via the MoLE config and reused. Predictions land in each task's configured `evaluation.predictions_file` path (the same one the per-task notebooks write).

In [ ]:
BATCH_SIZE = 16

for task_cfg, out_dir in [
    ('configs/e2e_gpt2_medium_lora.yaml',    'outputs/runs/e2e_lora_r4_alpha32_via_mole'),
    ('configs/dart_gpt2_medium_lora.yaml',   'outputs/runs/dart_lora_r4_alpha32_via_mole'),
    ('configs/webnlg_gpt2_medium_lora.yaml', 'outputs/runs/webnlg_lora_r4_alpha32_via_mole'),
]:
    !mkdir -p "$out_dir"
    print('=== task config:', task_cfg, '===')
    !TOKENIZERS_PARALLELISM=false TRANSFORMERS_VERBOSITY=error python scripts/generate_mole.py \
        --mole-config configs/mole_e2e_dart_webnlg.yaml \
        --task-config "$task_cfg" \
        --router "$LOCAL_ROUTER" \
        --split test \
        --batch-size "$BATCH_SIZE" \
        --output-file "$out_dir/generations_test.txt" 

## 13. Per-Task Quick Metrics

`scripts/evaluate.py` reads `evaluation.predictions_file`/`references_file` from each task's config. Override `--predictions` so it scores the routed model's outputs.

In [ ]:
for task_cfg, out_dir in [
    ('configs/e2e_gpt2_medium_lora.yaml',    'outputs/runs/e2e_lora_r4_alpha32_via_mole'),
    ('configs/dart_gpt2_medium_lora.yaml',   'outputs/runs/dart_lora_r4_alpha32_via_mole'),
    ('configs/webnlg_gpt2_medium_lora.yaml', 'outputs/runs/webnlg_lora_r4_alpha32_via_mole'),
]:
    print('=== task config:', task_cfg, '===')
    pred_file = f'{out_dir}/generations_test.txt'
    !python scripts/evaluate.py --config "$task_cfg" --predictions-file "$pred_file"
    metrics_file = f'{out_dir}/generations_test.metrics.json'
    !cat "$metrics_file" 

## 14. Ablation: Force-Expert Generation

Pins the gate to one expert at a time and regenerates. Forced expert k should reproduce the standalone single-adapter baseline for task k (this is the load-correctness check from `tests/test_mole.py` applied end-to-end). Significantly worse cross-task numbers (forced E2E expert evaluated on DART, etc.) confirm the experts specialized.

In [ ]:
ABLATION_TASK_CONFIG = 'configs/dart_gpt2_medium_lora.yaml'  # change as desired
ABLATION_OUT_DIR = 'outputs/runs/mole_ablation_dart'
!mkdir -p "$ABLATION_OUT_DIR"

for expert in ('e2e', 'dart', 'webnlg'):
    print('=== forcing expert:', expert, '===')
    pred_file = f'{ABLATION_OUT_DIR}/forced_{expert}.txt'
    !TOKENIZERS_PARALLELISM=false TRANSFORMERS_VERBOSITY=error python scripts/generate_mole.py \
        --mole-config configs/mole_e2e_dart_webnlg.yaml \
        --task-config "$ABLATION_TASK_CONFIG" \
        --force-expert "$expert" \
        --split test \
        --max-examples 200 \
        --output-file "$pred_file"
    !python scripts/evaluate.py --config "$ABLATION_TASK_CONFIG" --predictions-file "$pred_file" --max-examples 200
    !cat "${pred_file%.txt}.metrics.json" 

## 15. Final Drive Backup

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
for via in ('e2e_lora_r4_alpha32_via_mole', 'dart_lora_r4_alpha32_via_mole', 'webnlg_lora_r4_alpha32_via_mole', 'mole_ablation_dart'):
    backup_to_run(f'outputs/runs/{via}', destination_name=via)
backup_to_run('configs/mole_e2e_dart_webnlg.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_mole.ipynb')

print('Backed up complete MoLE run to:', DRIVE_RUN_DIR)
!find /content/drive/MyDrive/mole_e2e_dart_webnlg -maxdepth 3 -type f | sort | tail -60